# Google Merchandise Store — Product Analytics
Compact analysis of an e-commerce funnel, device segmentation, cohort retention, and a statistical comparison of purchase conversion.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.proportion import confint_proportions_2indep

## 1. Funnel

In [ ]:
funnel = pd.read_csv('../data/funnel.csv')
funnel

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(funnel['stage'], funnel['sessions'])
ax.invert_yaxis()
ax.set_xlabel('Sessions')
ax.set_title('E-commerce Funnel')
for i, v in enumerate(funnel['sessions']):
    ax.text(v, i, f' {v:,}', va='center')
plt.tight_layout()
plt.show()

**Key observation:** only 13.74% of all sessions reached a product view, and the largest drop within the product funnel was Product view → Add to cart (59.63% drop-off).

## 2. Device segmentation

In [ ]:
device = pd.read_csv('../data/device_funnel.csv')
device

In [ ]:
plot_df = device[device['device_category'] != 'ALL']
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(plot_df['device_category'], plot_df['session_to_purchase_pct'])
ax.set_ylabel('Session → purchase conversion, %')
ax.set_title('Purchase Conversion by Device')
for bar, val in zip(bars, plot_df['session_to_purchase_pct']):
    ax.text(bar.get_x()+bar.get_width()/2, val, f'{val:.2f}%', ha='center', va='bottom')
plt.tight_layout()
plt.show()

**Key observation:** desktop session-to-purchase conversion was 1.58%, versus 0.41% on mobile. The gap is visible at every lower stage of the funnel.

## 3. Cohort retention

In [ ]:
retention = pd.read_csv('../data/retention.csv', parse_dates=['cohort_month'])
retention.head()

In [ ]:
heat = retention.pivot(index='cohort_month', columns='month_number', values='retention_pct')
heat.index = heat.index.strftime('%Y-%m')
heat = heat.loc[:, [c for c in heat.columns if c > 0]]
fig, ax = plt.subplots(figsize=(12, 7))
im = ax.imshow(heat.values, aspect='auto')
ax.set_xticks(np.arange(len(heat.columns)))
ax.set_xticklabels([f'M{c}' for c in heat.columns])
ax.set_yticks(np.arange(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_xlabel('Months since first visit')
ax.set_ylabel('Cohort')
ax.set_title('Monthly Cohort Retention')
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.iloc[i, j]
        if pd.notna(val):
            ax.text(j, i, f'{val:.2f}%', ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, label='Retention, %')
plt.tight_layout()
plt.show()

In [ ]:
weighted = (retention[retention['month_number'] > 0]
            .groupby('month_number')
            .agg(active_users=('active_users','sum'), cohort_size=('cohort_size','sum'))
            .reset_index())
weighted['retention_pct'] = 100 * weighted['active_users'] / weighted['cohort_size']
weighted

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(weighted['month_number'], weighted['retention_pct'], marker='o')
ax.set_xticks(weighted['month_number'])
ax.set_xlabel('Months since first visit')
ax.set_ylabel('Weighted retention, %')
ax.set_title('Retention Curve')
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

**Key observation:** weighted retention falls from 3.39% at M1 to 1.09% at M2 and 0.58% at M3. M1 retention across cohorts ranged from 2.84% to 4.28%.

## 4. Statistical comparison: desktop vs mobile

In [ ]:
desktop_purchases, desktop_sessions = 10528, 664479
mobile_purchases, mobile_sessions = 856, 208725
count = np.array([desktop_purchases, mobile_purchases])
nobs = np.array([desktop_sessions, mobile_sessions])
z_stat, p_value = proportions_ztest(count, nobs)
desktop_cr = desktop_purchases / desktop_sessions
mobile_cr = mobile_purchases / mobile_sessions
diff = desktop_cr - mobile_cr
print(f'Desktop CR: {desktop_cr:.3%}')
print(f'Mobile CR: {mobile_cr:.3%}')
print(f'Absolute difference: {diff*100:.3f} pp')
print(f'Ratio: {desktop_cr/mobile_cr:.2f}x')
print(f'z = {z_stat:.2f}, p-value = {p_value:.3g}')

The difference is statistically significant, but this is **not an A/B test**: device groups were not randomized. Therefore, the result shows an observed segment difference and does not establish causality.

## Limitations
- Public, anonymized sample dataset.
- Device comparison is observational, not randomized.
- Later cohorts have shorter follow-up windows.
- The dataset ends with only one day of August 2017, so that date was excluded from retention cohorts.